# Kaggle ASR v1 Workflow
Single-run workflow: setup, full v1 pipeline, and artifact export.

## Pre-run in Kaggle UI
- Settings -> Accelerator: GPU (T4)
- Settings -> Internet: ON
- Add Input datasets: code dataset + raw dataset.
- Run Save Version -> Save & Run All (Commit) to persist v1 outputs.

In [ ]:
import glob
import os
import shutil
import zipfile
from pathlib import Path

workspace = Path('/kaggle/working/speech_recognation')
workspace.mkdir(parents=True, exist_ok=True)

# Case A: dataset je već raspakovan (npr. /kaggle/input/<dataset>/kaggle/requirements_kaggle.txt)
req_candidates = glob.glob('/kaggle/input/**/kaggle/requirements_kaggle.txt', recursive=True)
if req_candidates:
    project_root = Path(req_candidates[0]).parents[1]
    print('Found extracted project at:', project_root)
    shutil.copytree(project_root, workspace, dirs_exist_ok=True)
    os.chdir(workspace)
else:
    # Case B: dataset sadrži zip (npr. speech_recognation_code_for_kaggle.zip)
    zip_candidates = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if not zip_candidates:
        raise FileNotFoundError(
            'Nisam našao ni raspakovan projekat ni zip u /kaggle/input. Attach code dataset.'
        )

    preferred = [z for z in zip_candidates if 'speech_recognation_code_for_kaggle' in z]
    zip_path = preferred[0] if preferred else zip_candidates[0]
    print('Using zip:', zip_path)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(workspace)
    os.chdir(workspace)

assert Path('kaggle/requirements_kaggle.txt').exists(), 'Bootstrap failed: missing kaggle/requirements_kaggle.txt'
print('CWD:', Path.cwd())

In [ ]:
!python -m pip install -U pip
!python -m pip install -r kaggle/requirements_kaggle.txt
!python kaggle/prepare_environment.py

In [ ]:
from pathlib import Path
import glob

# If you know the exact path, set it here; leave None for auto-detection.
RAW_DIR = None
WORK_DIR_V1 = '/kaggle/working/asr_full_run_v1'

EPOCHS_V1 = 8
BATCH_SIZE_V1 = 8
LR_V1 = 2e-6
HOLDOUT_RATIO = 0.15

input_root = Path('/kaggle/input')
assert input_root.exists(), 'Missing /kaggle/input'

print('Datasets visible under /kaggle/input:')
for item in sorted(input_root.iterdir()):
    if item.is_dir():
        print(' -', item.name)

# Search recursively because Kaggle may nest datasets as:
# /kaggle/input/datasets/<owner>/<dataset>/...
recursive_raw = [Path(p) for p in glob.glob('/kaggle/input/**/raw', recursive=True)]

candidates = []
for raw_dir in recursive_raw:
    if raw_dir.is_dir():
        audio_count = sum(
            1
            for p in raw_dir.rglob('*')
            if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
        )
        text_count = sum(1 for p in raw_dir.rglob('*.txt'))
        if audio_count > 0 and text_count > 0:
            candidates.append((raw_dir, audio_count, text_count))

# Fallback: if no /raw directories are detected, scan any folder with both audio + txt
if not candidates:
    for p in [Path(x) for x in glob.glob('/kaggle/input/**', recursive=True)]:
        if not p.is_dir():
            continue
        audio_count = sum(
            1
            for f in p.rglob('*')
            if f.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
        )
        text_count = sum(1 for f in p.rglob('*.txt'))
        if audio_count > 0 and text_count > 0:
            candidates.append((p, audio_count, text_count))

if RAW_DIR is None:
    if not candidates:
        raise FileNotFoundError(
            'Auto-detection failed. Please set RAW_DIR manually to your raw dataset path.'
        )

    # Prefer paths containing speech-recognation-raw
    preferred = [x for x in candidates if 'speech-recognation-raw' in str(x[0])]
    selected_path, audio_count, text_count = (preferred[0] if preferred else candidates[0])
    RAW_DIR = str(selected_path.resolve())
else:
    RAW_DIR = RAW_DIR.strip()
    selected_path = Path(RAW_DIR)
    audio_count = sum(
        1
        for p in selected_path.rglob('*')
        if p.suffix.lower() in {'.mp3', '.wav', '.flac', '.m4a', '.ogg'}
    )
    text_count = sum(1 for p in selected_path.rglob('*.txt'))

assert Path(RAW_DIR).exists(), f'RAW_DIR does not exist: {RAW_DIR}'
print('Selected RAW_DIR:', RAW_DIR)
print(f'Found files under RAW_DIR -> audio: {audio_count}, text: {text_count}')

if audio_count == 0 or text_count == 0:
    raise ValueError(
        f'RAW_DIR is not valid for training (audio={audio_count}, text={text_count}).'
    )

In [ ]:
import subprocess

cmd = [
    'python', 'kaggle/run_full_pipeline.py',
    '--raw_dir', RAW_DIR,
    '--work_dir', WORK_DIR_V1,
    '--epochs', str(EPOCHS_V1),
    '--batch_size', str(BATCH_SIZE_V1),
    '--learning_rate', str(LR_V1),
    '--holdout_ratio', str(HOLDOUT_RATIO),
]
print('RUN:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import json
from pathlib import Path

summary_v1 = Path(WORK_DIR_V1) / 'metrics' / 'comparison_summary.json'
print(json.dumps(json.loads(summary_v1.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
!python kaggle/export_artifacts.py --source_dir /kaggle/working/asr_full_run_v1 --zip_prefix asr_full_run_v1